# Phutball transfer suite (llm_transfer)

Does phutball training carry over? The same suite runs on the base model, the self-stop warm start, the v4 curriculum
best (r14) and the gated-GRPO best, and `llm_transfer.report` compares them item by item (paired z) per family and
difficulty dial. Families, ordered by distance from the training task: **bridge** (the validation win puzzles rewritten
without the board: same logic, new surface; eval only, never trained on), **pointer** chasing (1-8 hops, broken chains),
**path** finding (no-route cases), **boxes** (state tracking, REMOVE-heavy half), **ruler_vt** (variable tracking in
2k / 8k tokens of filler), **babilong** (qa2 / qa3 at 0k-8k, qa19 path finding), **cogniload** (20-100 statements),
**core** (code data / control / information-flow dependence, yes/no). The prediction to test: gains that grow with
hops / depth, most on REMOVE-heavy tracking, and fewer false "none" answers.
**Runtime:** A100. Roughly 1,400 items per model, ~40-60 min each.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, subprocess, time, urllib.request
OUT = '/content/drive/MyDrive/phutball/transfer'; os.makedirs(OUT, exist_ok=True)
MODELS = {                                                 # name -> LoRA adapter (None: the base model)
    'base': None,
    'warm': '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final',
    'r14': '/content/drive/MyDrive/phutball/curriculum_v4/r14/adapter/final',
    'grpo': '/content/drive/MyDrive/phutball/grpo_cost/p3/final',   # update to the gated-GRPO best (gate_state.json 'best')
}
MAXLEN = 20000; CAP = 6144
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
%cd /content/phutball-jax
!python -m llm_transfer.build --out llm_transfer/data/suite_v1.jsonl
adapters = {k: v for k, v in MODELS.items() if v}
!pkill -f "vllm serve" ; sleep 5
log = open('/content/vllm_serve.log', 'w')
p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(MAXLEN), '--gpu-memory-utilization', '0.90',
                      '--port', '8000', '--enable-lora', '--max-lora-rank', '32', '--max-loras', str(len(adapters)),
                      '--lora-modules', *[f'{k}={v}' for k, v in adapters.items()]], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try: urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up'); break
    except Exception: time.sleep(5)
else: raise RuntimeError(open('/content/vllm_serve.log').read()[-2000:])

In [ ]:
for name in MODELS:
    if os.path.exists(f'{OUT}/{name}.summary.json'): print('done:', name); continue
    model = 'Qwen/Qwen3.5-9B' if MODELS[name] is None else name
    !python -m llm_transfer.run --model {model} --name {name} --max-tokens {CAP} --concurrency 64 | tail -60
    !cp llm_transfer/results/{name}.jsonl llm_transfer/results/{name}.summary.json {OUT}/

In [ ]:
!mkdir -p llm_transfer/results && cp {OUT}/*.jsonl {OUT}/*.summary.json llm_transfer/results/
names = ' '.join(n for n in MODELS if os.path.exists(f'{OUT}/{n}.summary.json'))
EV4 = '/content/drive/MyDrive/phutball/curriculum_v4/eval'; EVG = '/content/drive/MyDrive/phutball/grpo_cost/eval'
!python -m llm_transfer.report {names} --phutball r14={EV4}/v14 grpo={EVG}/v3
!python -m llm_transfer.report {names} --key correct

In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()